In [173]:
import pandas as pd
import re
import json
from tqdm import tqdm

### Loading Turkey-Syria, 2023 Tweets Dataset

In [2]:
df = pd.read_csv("../datasets/tweets.csv")
df.drop(df[df['language'] != 'en'].index, inplace=True)

C:\Users\deepp\AppData\Local\Temp\ipykernel_32636\3501300992.py:1: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../datasets/tweets.csv")


### Percentage of Geo-tagged tweets

In [3]:
(100*df['place'].notnull().sum())/(df['content'].notnull().sum())


5.014607701475536

### Extracting Hashtags

In [179]:
def process_hashtags(hashtags):
    if isinstance(hashtags, str): 
        return [tag.strip().strip(']').strip('[').strip("'") for tag in hashtags.split(",")]
    return [] 

jsonl_data = []
all_gpes = set()

for index, row in df.iterrows():
    content = row["content"]
    hashtags = process_hashtags(row["hashtags"])
    entities = []
    
    for hashtag in hashtags:
        all_gpes.add(hashtag)

hashtags = pd.DataFrame(list(all_gpes), columns=["GPE"])
hashtags.to_csv("hashtags.csv", index=False)


In [166]:
gpe = pd.read_csv('hashtags.csv')

In [167]:
gpe.sort_values(by="GPE", ascending=True, inplace=True)

In [168]:
import string
gpe = gpe[gpe["GPE"].str.len() >= 4]
def is_valid(entry):
    return not any(char in string.punctuation + "@" + "#" or char.isdigit() for char in entry)

gpe = gpe[gpe["GPE"].apply(is_valid)]

In [169]:
gpe = gpe.drop_duplicates(subset="GPE")


In [13]:
gpe.to_csv('gpe.csv', index = False)

In [14]:
geonames_df = pd.read_csv('../datasets/geonames.csv')

C:\Users\deepp\AppData\Local\Temp\ipykernel_1892\658826672.py:1: DtypeWarning: Columns (9,10,11,12,13) have mixed types. Specify dtype option on import or set low_memory=False.
  geonames_df = pd.read_csv('../datasets/geonames.csv')


In [16]:
jp = geonames_df[geonames_df['country code']=='JP']

In [18]:
jp.to_csv('../datasets/jp.csv')

In [75]:
city = geonames_df[(geonames_df['feature class'].isin(['A'])) & (geonames_df['population'] > 5000000)]

In [76]:
city.to_csv('../datasets/city.csv')

In [170]:
gpe_df = pd.read_csv('gpe.csv')

In [22]:
geonames_df = geonames_df[geonames_df['population']>0]

### 2-Step Validation of GPEs

In [24]:
valid_gpe = gpe_df[gpe_df['GPE'].isin(geonames_df['name'])]

In [25]:
valid_gpe_df = valid_gpe.reset_index(drop=True)
valid_gpe_df.to_csv('valid_gpe.csv', index=False)

In [151]:
import pandas as pd
from opencage.geocoder import OpenCageGeocode
import time

api_key = '################################'

geocoder = OpenCageGeocode(api_key)

valid_gpe_df = pd.read_csv('valid_gpe.csv')

def verify_location(location):
    results = geocoder.geocode(location)
    if results:
        return True
    return False

valid_locations = []
for location in valid_gpe_df['GPE']:  
    if verify_location(location):
        valid_locations.append(location)
    else:
        print(f"Invalid location: {location}")
    time.sleep(1)

valid_locations_df = pd.DataFrame(valid_locations, columns=['valid_gpe'])

valid_locations_df.to_csv('verified_valid_gpe.csv', index=False)

print(valid_locations_df.head())


  valid_gpe
0   Aalborg
1  Abeokuta
2  Abkhazia
3    Abrams
4     Abuja


In [171]:
valid_locations = pd.read_csv('verified_valid_gpe.csv')

In [82]:
valid_locations.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1896 entries, 0 to 1895
Data columns (total 1 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   valid_gpe  1896 non-null   object
dtypes: object(1)
memory usage: 14.9+ KB


In [198]:
countries = pd.read_csv('../datasets/world-data-2023.csv')

In [199]:
countries['Population'] = (
    countries['Population']
    .str.replace(",", "", regex=True)  
    .apply(pd.to_numeric, errors='coerce')  
    .astype("Int64") )

countries['Population'].fillna(0, inplace=True)  
print(countries['Population'].head())

0    38041754
1     2854191
2    43053054
3       77142
4    31825295
Name: Population, dtype: Int64


In [200]:
countries = countries[countries['Population'] > 6000000]

In [201]:
countries.to_csv('../datasets/countries.csv')